In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

### If you are using the python-dotenv library and called load_dotenv() at the top of your file, 
# that function automatically finds your .env file and seamlessly injects everything inside it directly into os.environ for you.


True

In [2]:
# os.getenv(key = "LANGCHAIN_AI_KEY") # This function is for getting the values of keys present inside your .env file
# This has nothing to do with the environment variables of your computer.

In [3]:
type(os.environ) # This contains all the environment variables present inside your computer

# when you write os.environ["OPENAI_API_KEY"] = os.getenv(key = "OPENAI_API_KEY"), you are basically telling your 
# computer to look for OPENAI_API_KEY inside your .env file and then load that value temporarily inside the environment
# variables of your OS


# But since we ran the load_dotenv() initially, we dont need to do  this redundant step

os._Environ

# Creating Vector Embeddings

In [4]:
from langchain_openai import OpenAIEmbeddings
embedder = OpenAIEmbeddings(model = "text-embedding-3-small")
print(embedder)

text = "This will be a tutorial on OPENAI Embeddings"
vectorized_text = embedder.embed_query(text)
len(vectorized_text)
# we get a vector of size 1536



c:\Users\Abhishek Jha\miniconda3\envs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


client=<openai.resources.embeddings.Embeddings object at 0x0000011F5A2444C0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000011F5A2455D0> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


1536

In [5]:
### we can create vectors of our own preferred size as well.
from langchain_openai import OpenAIEmbeddings

embedder_1024 = OpenAIEmbeddings(model = "text-embedding-3-large", dimensions=1024)
vectorized_text_1024 = embedder_1024.embed_query(text)
len(vectorized_text_1024)

1024

# Storing in Database

When generating vectors with an embedding model, the next step is storing them in a vector database. 

Usually, this happens in a single step: we pass the documents and the embedder directly to the database, which automatically converts the text into vectors and stores them.

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings

# step 1 (loading)
text_loader = TextLoader("speech.txt")
initial_docs = text_loader.load()

# step 2 (making document chunks)
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 50)
final_docs = splitter.split_documents(initial_docs) # 45 docs

# Now store all this in Vector Database

embedder = OpenAIEmbeddings(model = "text-embedding-3-large", dimensions = 3000)

In [7]:
# store in database
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(final_docs, embedder)

In [8]:
len(db)

45

In [9]:
query = "what happened to the heavy oak doors?"

retrieved_results = db.similarity_search(query = query)

print(len(retrieved_results))

4


In [10]:
for result in retrieved_results:
    print(result.page_content + "\n\n\n")

The heavy oak doors of the auditorium sealed shut, plunging the crowded hall into a hushed,



home with a sudden, thunderous crescendo that practically vibrated the floorboards beneath their



shut, plunging the crowded hall into a hushed, expectant silence just before Elias approached the



silence just before Elias approached the mahogany podium, his footsteps echoing like a slow,





# Using ollama Embeddings

In [11]:
from langchain_ollama import OllamaEmbeddings

ollama_embedder = OllamaEmbeddings(model="llama3.2:1b", dimensions=1000)
ollama_vectorized_text = ollama_embedder.embed_query(text)
len(ollama_vectorized_text)

1000

In [12]:
embedded_greek_sentences = ollama_embedder.embed_documents(
                                ["The first letter of the Greek alphabet is Alpha.",
                                 "The second letter of the Greek alphabet is Beta."]
                                 )

len(embedded_greek_sentences), len(embedded_greek_sentences[0]), len(embedded_greek_sentences[1])

(2, 1000, 1000)

#### To view other embedding models, visit https://ollama.com/blog/embedding-models

In [13]:
from langchain_ollama import OllamaEmbeddings

ollama_embedder_2 = OllamaEmbeddings(model = "mxbai-embed-large", dimensions = 900)
embedded_text_ollama_2 = ollama_embedder_2.embed_query(text)
len(embedded_text_ollama_2)

900

In [14]:
greek_embeds = ollama_embedder_2.embed_documents(["The first letter of the Greek alphabet is Alpha.",
                                   "The second letter of the Greek alphabet is Beta."])
len(greek_embeds), len(greek_embeds[0]), len(greek_embeds[1])

(2, 900, 900)

# Using HuggingFace

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
embedder = HuggingFaceEmbeddings(model = "all-MiniLM-L6-v2")
hf_embed_text = embedder.embed_query(text)
len(hf_embed_text)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4564.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384